# 🛰️ ISRO Satellite Dataset — LangChain Insights Pipeline
**End-to-end: Data Loading → Filtering → Graphs → Tools → LCEL Chains → ReAct Agent → 100-word Insight**

Stack: OpenRouter (free LLM) · LangChain · LangGraph · Pandas · Matplotlib

---

## 1. Install Packages

In [ ]:
!pip install -q \
    langchain langchain-openai langchain-community langchain-core \
    langchain-huggingface langgraph chromadb pydantic \
    sentence-transformers matplotlib pandas

## 2. Configure Free LLM (OpenRouter)
Sign up at https://openrouter.ai — no credit card needed.

In [ ]:
import os
from langchain_openai import ChatOpenAI

# ── Option A: paste key directly ──
OPENROUTER_API_KEY = "sk-or-..."   # <-- Replace with your key

# ── Option B: Colab Secrets (recommended) ──
# from google.colab import userdata
# OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')

os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

llm = ChatOpenAI(
    model="meta-llama/llama-3.3-70b-instruct:free",
    temperature=0,
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
)

print(llm.invoke("Say: LLM is ready!").content)
print("✅ LLM configured")

## 3. Load & Filter ISRO Dataset
Upload `ISRO_Satellite_Dataset.csv` to Colab before running this cell.

In [ ]:
import pandas as pd

# Upload via Colab UI or uncomment below:
# from google.colab import files
# uploaded = files.upload()

df = pd.read_csv("ISRO_Satellite_Dataset.csv")
df["Year"] = pd.to_datetime(df["Date of Launch"], errors="coerce").dt.year

print(f"Shape     : {df.shape}")
print(f"Columns   : {df.columns.tolist()}")
print(f"Date range: {df['Year'].min():.0f} – {df['Year'].max():.0f}")
df.head(3)

In [ ]:
# ── Data Filtering & Aggregations ──
df_clean = df.dropna(subset=["Launch Mass (kg.)", "Expected Lifetime (yrs.)",
                              "Class of Orbit", "Purpose"])

purpose_counts   = df["Purpose"].value_counts().to_dict()
orbit_counts     = df["Class of Orbit"].value_counts().to_dict()
launches_by_year = df["Year"].value_counts().sort_index().to_dict()
avg_mass_orbit   = df.groupby("Class of Orbit")["Launch Mass (kg.)"].mean().to_dict()
avg_life_purpose = df.groupby("Purpose")["Expected Lifetime (yrs.)"].mean().to_dict()
vehicle_counts   = df["Launch Vehicle"].value_counts().to_dict()

print("Purpose counts :", purpose_counts)
print("Orbit counts   :", orbit_counts)
print("Avg mass/orbit :", {k: round(v, 1) for k, v in avg_mass_orbit.items()})
print("Avg lifetime   :", {k: round(v, 1) for k, v in avg_life_purpose.items()})
print("\n✅ Data filtered and aggregated")

## 4. Visualize — 4 Key Charts

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("ISRO Satellite Dataset — Key Insights", fontsize=14, fontweight="bold")

# ── Chart 1: Purpose donut ──
ax = axes[0, 0]
colors = ["#378ADD", "#1D9E75", "#EF9F27", "#888780", "#D85A30", "#533489"]
ax.pie(
    list(purpose_counts.values()),
    labels=list(purpose_counts.keys()),
    autopct="%1.0f%%",
    colors=colors[:len(purpose_counts)],
    wedgeprops={"width": 0.55},
    startangle=90
)
ax.set_title("Satellites by Purpose")

# ── Chart 2: Launches per year ──
ax = axes[0, 1]
ax.bar(list(launches_by_year.keys()), list(launches_by_year.values()),
       color="#378ADD", edgecolor="white")
ax.set_title("Launches per Year")
ax.set_xlabel("Year")
ax.set_ylabel("Count")
ax.tick_params(axis="x", rotation=45)

# ── Chart 3: Avg launch mass by orbit ──
ax = axes[1, 0]
orbits = list(avg_mass_orbit.keys())
masses = [round(v, 0) for v in avg_mass_orbit.values()]
bars = ax.barh(orbits, masses, color=["#533489", "#1D9E75"])
ax.set_title("Avg Launch Mass by Orbit (kg)")
ax.set_xlabel("kg")
for bar, val in zip(bars, masses):
    ax.text(bar.get_width() - 80, bar.get_y() + bar.get_height() / 2,
            f"{int(val):,} kg", va="center", ha="right", color="white", fontsize=9)

# ── Chart 4: Avg lifetime by purpose ──
ax = axes[1, 1]
ax.bar(list(avg_life_purpose.keys()),
       [round(v, 1) for v in avg_life_purpose.values()],
       color="#EF9F27", edgecolor="white")
ax.set_title("Avg Expected Lifetime by Purpose (yrs)")
ax.set_ylabel("Years")
ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig("isro_insights.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved as isro_insights.png")

## 5. Define LangChain Tools
Each `@tool` wraps a dataset aggregation. The LLM reads the docstring to decide when to call it.

> Pattern from notebook Section 7: *Tools — Giving LLMs Superpowers*

In [ ]:
from langchain_core.tools import tool

@tool
def get_purpose_stats(purpose: str = "all") -> str:
    """Get satellite count by purpose.
    Pass a specific purpose like 'Earth Observation' or 'all' for full breakdown."""
    if purpose.lower() == "all":
        return str(purpose_counts)
    count = purpose_counts.get(purpose, 0)
    return f"{purpose}: {count} satellites"

@tool
def get_orbit_mass(orbit: str = "all") -> str:
    """Get average launch mass for GEO or LEO orbit class in kg.
    Pass 'GEO', 'LEO', or 'all'."""
    if orbit.lower() == "all":
        return str({k: round(v, 1) for k, v in avg_mass_orbit.items()})
    val = avg_mass_orbit.get(orbit.upper())
    return f"Avg launch mass for {orbit}: {round(val, 1)} kg" if val else f"Orbit '{orbit}' not found"

@tool
def get_lifetime_by_purpose(purpose: str = "all") -> str:
    """Get average expected satellite lifetime in years for a given purpose.
    Pass purpose name or 'all'."""
    if purpose.lower() == "all":
        return str({k: round(v, 2) for k, v in avg_life_purpose.items()})
    val = avg_life_purpose.get(purpose)
    return f"Avg lifetime for {purpose}: {round(val, 1)} yrs" if val else "Purpose not found"

@tool
def get_launch_vehicle_stats() -> str:
    """Get the count of satellites launched by each launch vehicle (PSLV, GSLV, Ariane etc.)"""
    return str(vehicle_counts)

@tool
def get_yearly_trend(start_year: int = 2003, end_year: int = 2020) -> str:
    """Get number of ISRO satellite launches per year between start_year and end_year."""
    filtered = {k: v for k, v in launches_by_year.items() if start_year <= k <= end_year}
    return str(dict(sorted(filtered.items())))

# Inspect a tool — the LLM reads these docstrings!
print(f"Tool name  : {get_purpose_stats.name}")
print(f"Description: {get_purpose_stats.description}")
print()
print("Test call:", get_orbit_mass.invoke("GEO"))
print("Test call:", get_lifetime_by_purpose.invoke("Communications"))

## 6. Bind Tools to LLM
> Pattern from notebook Section 7: `llm.bind_tools(tools)` — LLM now *chooses* to call a tool.

In [ ]:
tools = [get_purpose_stats, get_orbit_mass, get_lifetime_by_purpose,
         get_launch_vehicle_stats, get_yearly_trend]

llm_with_tools = llm.bind_tools(tools)

# The LLM decides which tool to call
response = llm_with_tools.invoke("What is the average mass of a GEO satellite in the ISRO fleet?")
print("Content    :", response.content)
print("Tool calls :", response.tool_calls)

## 7. Manual Tool Execution Loop
> Pattern from notebook Section 7 (Cell 24): *understand what happens INSIDE an agent.*

Steps: LLM decides → we execute tool → feed result back → LLM gives final answer.

In [ ]:
from langchain_core.messages import HumanMessage, ToolMessage

def run_tool_loop(question: str):
    print(f"\nQ: {question}")
    print("-" * 50)

    # Step 1 — LLM picks a tool
    ai_msg = llm_with_tools.invoke(question)
    if not ai_msg.tool_calls:
        print("Direct answer:", ai_msg.content)
        return

    # Step 2 — Execute tool
    tool_map = {t.name: t for t in tools}
    tool_results = []
    for tc in ai_msg.tool_calls:
        result = tool_map[tc["name"]].invoke(tc["args"])
        print(f"  [Tool] {tc['name']} → {result[:120]}")
        tool_results.append(ToolMessage(content=result, tool_call_id=tc["id"]))

    # Step 3 — Feed result back
    messages = [HumanMessage(content=question), ai_msg] + tool_results
    final = llm_with_tools.invoke(messages)
    print(f"  Answer: {final.content}")

run_tool_loop("How many Earth Observation satellites does ISRO have?")
run_tool_loop("Which launch vehicle has been used most by ISRO?")
run_tool_loop("What is the average lifetime of a Communications satellite?")

## 8. LCEL Chain Layers — prompt | llm | parser
> Pattern from notebook Sections 4 & 4 (Cells 13–14): *compose with the pipe operator.*

**Flow:** Stats input → Layer 1 (summarize chain) → Layer 2 (insight chain) → Final answer

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ── LAYER 1: Summarize stats ──
summarize_prompt = ChatPromptTemplate.from_template(
    """You are a space data analyst. Given these ISRO satellite statistics:
- Purpose distribution: {purpose_counts}
- Orbit distribution: {orbit_counts}
- Avg launch mass by orbit (kg): {avg_mass}
- Avg lifetime by purpose (yrs): {avg_lifetime}
- Top launch vehicles: {vehicles}

Summarize the 3 most important findings in bullet points. Be concise."""
)
summarize_chain = summarize_prompt | llm | StrOutputParser()

# ── LAYER 2: Generate insight from summary ──
insight_prompt = ChatPromptTemplate.from_template(
    """Based on this ISRO satellite data summary:
{summary}

Write a strategic insight in exactly 100 words about ISRO's satellite program."""
)
insight_chain = insight_prompt | llm | StrOutputParser()

# ── COMPOSE: Layer 1 feeds into Layer 2 ──
full_pipeline = (
    summarize_chain
    | (lambda summary: {"summary": summary})
    | insight_chain
)

stats_input = {
    "purpose_counts": str(purpose_counts),
    "orbit_counts": str(orbit_counts),
    "avg_mass": str({k: round(v, 0) for k, v in avg_mass_orbit.items()}),
    "avg_lifetime": str({k: round(v, 1) for k, v in avg_life_purpose.items()}),
    "vehicles": str(dict(list(vehicle_counts.items())[:6]))
}

print("Running LCEL pipeline: Layer 1 → Layer 2 ...\n")
final_insight = full_pipeline.invoke(stats_input)

print("=" * 60)
print(final_insight)
print("=" * 60)

## 9. ReAct Agent — Full Autopilot
> Pattern from notebook Section 8 (Cell 27): `create_react_agent` automates the think → act → observe loop.

In [ ]:
from langgraph.prebuilt import create_react_agent

agent = create_react_agent(llm, tools)

def ask_agent(question: str):
    print(f"\n{'=' * 60}")
    print(f"Q: {question}")
    print("=" * 60)
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    for msg in result["messages"]:
        if msg.type == "ai" and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  [Tool] {tc['name']}({tc['args']})")
        elif msg.type == "tool":
            print(f"  [Result] {msg.content[:120]}")
        elif msg.type == "ai" and msg.content:
            print(f"\nAnswer: {msg.content}")

ask_agent("What percentage of ISRO satellites are Earth Observation vs Communications?")
ask_agent("Compare average mass and lifetime of GEO vs LEO satellites.")
ask_agent("In which year did ISRO launch the most satellites and how many?")

## 10. Final 100-Word Insight
Single LCEL chain — structured prompt → LLM → clean string output.

In [ ]:
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a space industry analyst. Answer in exactly 100 words. Be precise and strategic."),
    ("human", """Using these ISRO satellite statistics, write a 100-word strategic insight:

Purpose distribution: {purpose_counts}
Orbit distribution: {orbit_counts}
Avg launch mass — GEO: {geo_mass:.0f} kg, LEO: {leo_mass:.0f} kg
Avg lifetime — Communications: {comms_life:.1f} yrs, Earth Observation: {eo_life:.1f} yrs
Peak launch year: 2016 (9 launches)
Primary launch vehicle: PSLV
Total satellites: 46 (2003–2020)""")
])

final_chain = final_prompt | llm | StrOutputParser()

answer = final_chain.invoke({
    "purpose_counts": str(purpose_counts),
    "orbit_counts": str(orbit_counts),
    "geo_mass": avg_mass_orbit.get("GEO", 0),
    "leo_mass": avg_mass_orbit.get("LEO", 0),
    "comms_life": avg_life_purpose.get("Communications", 0),
    "eo_life": avg_life_purpose.get("Earth Observation", 0),
})

print("\n" + "=" * 60)
print("FINAL 100-WORD INSIGHT")
print("=" * 60)
print(answer)
print("=" * 60)
print(f"\nWord count: {len(answer.split())}")

---
## Pipeline Summary

| Layer | Component | Role |
|-------|-----------|------|
| 1 | CSV Loader + Pandas | Load & filter 46 satellite records |
| 2 | Matplotlib | 4 insight charts |
| 3 | `@tool` decorator | Wrap aggregations as LLM-callable tools |
| 4 | `llm.bind_tools()` | LLM selects tools dynamically |
| 5 | Manual tool loop | Understand agent internals |
| 6 | LCEL `prompt | llm | parser` | Multi-step summarize → insight chain |
| 7 | ReAct Agent (LangGraph) | Autopilot tool-calling loop |
| 8 | Final 100-word chain | Structured strategic output |

**Total cost: $0.00** — OpenRouter free tier used throughout.

*Based on IIST Agentic AI Training Program — LangChain Complete Guide (Days 8–9)*